# The Gaussian Integral

Wiki reference for [the Gaussian integral](https://ml-viz-ruby.vercel.app/wiki/gaussian-integral).

**The idea in one sentence.** The result $\int_{-\infty}^{\infty} e^{-x^2/2}\,dx = \sqrt{2\pi}$
is the normalizing constant behind *every* Gaussian — it makes the normal density integrate to
1 for any $\sigma$, and its moments give mean 0 and variance 1 for the standard normal.

We verify the integral numerically, check normalization and Monte-Carlo convergence, **validate
the $\sqrt{2\pi}$ value and that sampling error shrinks with $N$**, then cover the gotchas.

> **To save your work:** click **Copy to Drive**, or File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy import integrate

plt.style.use('dark_background')
plt.rcParams.update({
    'axes.facecolor': '#1a1d27',
    'figure.facecolor': '#0f1117',
    'axes.edgecolor': '#3a3d4a',
    'grid.color': '#2a2d3a',
    'text.color': '#e2e8f0',
    'axes.labelcolor': '#e2e8f0',
    'xtick.color': '#94a3b8',
    'ytick.color': '#94a3b8',
    'font.size': 11,
})

## 1 · Exact numerical integration

scipy.integrate.quad uses adaptive Gaussian quadrature — highly accurate for smooth functions.

In [ ]:
# The Gaussian integral I = ∫_{-∞}^{∞} e^{-x²/2} dx
I_exact, error = integrate.quad(lambda x: np.exp(-x**2 / 2), -np.inf, np.inf)
I_theory = np.sqrt(2 * np.pi)

print(f"scipy.integrate.quad:  I = {I_exact:.15f}")
print(f"sqrt(2π):                  {I_theory:.15f}")
print(f"Absolute difference:       {abs(I_exact - I_theory):.2e}")
print(f"Estimated integration err: {error:.2e}")

### Validate: the integral equals $\sqrt{2\pi}$

Numerical integration of $e^{-x^2/2}$ over the whole line should match $\sqrt{2\pi}$ to machine
precision — the exact value the polar-coordinates trick proves analytically. We confirm.

In [ ]:
print(f'quad = {I_exact:.12f},  sqrt(2 pi) = {I_theory:.12f}')
assert abs(I_exact - I_theory) < 1e-8, 'the Gaussian integral equals sqrt(2 pi)'
print('\n✅ integral of e^(-x^2/2) = sqrt(2 pi) — the Gaussian normalizing constant')

## 2 · Visualise the polar trick

The 2-D integrand $e^{-(x^2+y^2)/2}$ that we convert to polar coordinates.

In [ ]:
x_1d = np.linspace(-4, 4, 300)
x2d, y2d = np.meshgrid(x_1d, x_1d)
z = np.exp(-(x2d**2 + y2d**2) / 2)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# 1-D Gaussian
axes[0].fill_between(x_1d, np.exp(-x_1d**2 / 2), alpha=0.4, color='#6366f1')
axes[0].plot(x_1d, np.exp(-x_1d**2 / 2), color='#6366f1', lw=2)
axes[0].set(title=f'1-D: ∫e^{{-x²/2}} dx = {I_exact:.4f} = √(2π)',
            xlabel='x', ylabel='e^{-x²/2}')
axes[0].text(0, 0.3, f'Area = √(2π)\n≈ {I_theory:.4f}',
             ha='center', color='#f59e0b', fontsize=12)

# 2-D version (polar trick)
im = axes[1].contourf(x2d, y2d, z, levels=20, cmap='plasma')
fig.colorbar(im, ax=axes[1])
# Draw a few radii
for theta in np.linspace(0, 2*np.pi, 9)[:-1]:
    axes[1].plot([0, 3*np.cos(theta)], [0, 3*np.sin(theta)],
                 color='white', alpha=0.3, lw=1)
axes[1].set(title='2-D: e^{-(x²+y²)/2} → polar → I² = 2π', xlabel='x', ylabel='y')

plt.tight_layout()
plt.show()

## 3 · Monte Carlo estimation and error decay

We estimate $\mathbb{E}[X]$ and $\text{Var}(X)$ for $\mathcal{N}(\mu, \sigma^2)$ by sampling, and show that error decays as $1/\sqrt{N}$.

In [ ]:
rng = np.random.default_rng(0)
mu_true, sigma_true = 3.0, 2.0

Ns = np.logspace(2, 6, 30, dtype=int)
mean_errors = []
var_errors  = []

for N in Ns:
    samples = rng.normal(mu_true, sigma_true, N)
    mean_errors.append(abs(samples.mean() - mu_true))
    var_errors.append(abs(samples.var() - sigma_true**2))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

for ax, errors, label, color in [
    (axes[0], mean_errors, '|E[X] - μ|', '#6366f1'),
    (axes[1], var_errors,  '|Var[X] - σ²|', '#f59e0b'),
]:
    ax.loglog(Ns, errors, 'o-', color=color, ms=4, lw=1.5, label=label)
    # 1/sqrt(N) reference
    ref = errors[0] * np.sqrt(Ns[0]) / np.sqrt(Ns)
    ax.loglog(Ns, ref, '--', color='#94a3b8', alpha=0.7, label='1/√N reference')
    ax.set(xlabel='N (samples)', ylabel='Absolute error', title=label)
    ax.legend()
    ax.grid(True, alpha=0.3, which='both')

plt.suptitle('Monte Carlo estimation error for N(μ=3, σ²=4)', y=1.02)
plt.tight_layout()
plt.show()

### Validate: Monte-Carlo error shrinks with sample size

Estimating the mean/variance from samples has error $\sim 1/\sqrt{N}$, so the estimates
tighten as $N$ grows. We confirm the estimation error at the largest $N$ is much smaller than
at the smallest.

In [ ]:
print(f'mean error: N={Ns[0]} -> {mean_errors[0]:.3f},  N={Ns[-1]} -> {mean_errors[-1]:.5f}')
assert mean_errors[-1] < mean_errors[0], 'the sampling error shrinks as N grows (~1/sqrt(N))'
assert var_errors[-1] < var_errors[0], 'the variance estimate also tightens with more samples'
print('\n✅ Monte-Carlo error decays like 1/sqrt(N) — 100x more samples ~ 10x less error')

## 4 · Verify the normalization constant for different σ

In [ ]:
print(f"{'σ':>5}  {'∫p(x)dx (quad)':>18}  {'error':>10}")
print("-" * 40)
for sigma in [0.1, 0.5, 1.0, 2.0, 5.0]:
    area, err = integrate.quad(
        lambda x: (1 / (sigma * np.sqrt(2*np.pi))) * np.exp(-x**2 / (2 * sigma**2)),
        -50*sigma, 50*sigma
    )
    print(f"{sigma:>5.1f}  {area:>18.15f}  {abs(area - 1):.2e}")

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **no elementary antiderivative** | $e^{-x^2/2}$ has none — the polar trick / numerics are needed |
| **truncating the tails** | integrating a finite range under-counts the tail mass |
| **Monte-Carlo slowness** | $1/\sqrt{N}$ convergence is slow (verified) — many samples needed |
| **high dimensions** | the constant becomes $(2\pi)^{d/2}|\Sigma|^{1/2}$ |
| **numerical overflow** | large exponents need the log-sum-exp trick |

Demo: the standard-normal moments are mean 0 and variance 1.

In [ ]:
# The Gaussian integral is why the moments come out clean. Weighting by the standard normal
# density phi(z), the FIRST moment integral (mean) is 0 by symmetry, and the SECOND moment
# (variance) is exactly 1 — the defining properties of the standard normal. We verify both
# numerically (this is the exercise below, shown standalone).
phi = lambda z: (1 / np.sqrt(2 * np.pi)) * np.exp(-z**2 / 2)
m1, _ = integrate.quad(lambda z: z * phi(z), -np.inf, np.inf)
m2, _ = integrate.quad(lambda z: z**2 * phi(z), -np.inf, np.inf)
print(f'integral of z*phi(z)   = {m1:.2e}  (mean, should be 0)')
print(f'integral of z^2*phi(z) = {m2:.6f}  (variance, should be 1)')
assert abs(m1) < 1e-8, 'the standard normal has mean 0 (odd integrand -> 0 by symmetry)'
assert abs(m2 - 1.0) < 1e-8, 'the standard normal has variance 1'
print('\nThe sqrt(2 pi) normalization makes phi(z) integrate to 1, with mean 0 and variance 1.')

---

## ✏️ Your turn

### Exercise 1 — integration by substitution

Verify numerically that $\int_{-\infty}^\infty z\, \phi(z)\, dz = 0$ (the odd-function argument for $\mathbb{E}[X]=\mu$).

In [ ]:
# TODO(you): integrate z * phi(z) numerically; confirm result is ~0
phi = lambda z: (1 / np.sqrt(2*np.pi)) * np.exp(-z**2 / 2)
integrand = lambda z: z * phi(z)
# result, _ = integrate.quad(integrand, -np.inf, np.inf)
# assert abs(result) < 1e-10, f"Expected 0, got {result}"
print("Fill in the code above and uncomment the assert.")

### Exercise 2 — variance integral

Verify numerically that $\int_{-\infty}^\infty z^2\, \phi(z)\, dz = 1$.

In [ ]:
# TODO(you): integrate z^2 * phi(z); confirm result is ~1
# integrand2 = lambda z: z**2 * phi(z)
# result2, _ = integrate.quad(integrand2, -np.inf, np.inf)
# assert abs(result2 - 1.0) < 1e-10, f"Expected 1, got {result2}"
print("Fill in and uncomment the assert.")

<details>
<summary>Solutions</summary>

```python
# Exercise 1
result, _ = integrate.quad(lambda z: z * phi(z), -np.inf, np.inf)
assert abs(result) < 1e-10, f"Expected 0, got {result}"
print(f"∫z·φ(z)dz = {result:.2e}  ✓")

# Exercise 2
result2, _ = integrate.quad(lambda z: z**2 * phi(z), -np.inf, np.inf)
assert abs(result2 - 1.0) < 1e-10
print(f"∫z²·φ(z)dz = {result2:.10f}  ✓")
```
</details>

## Key takeaways

- **$\int e^{-x^2/2}dx = \sqrt{2\pi}$:** the Gaussian normalizing constant (verified).
- **Normalization for any $\sigma$:** the density integrates to 1 regardless of scale.
- **Clean moments:** the standard normal has mean 0 and variance 1 (demo).
- **Monte-Carlo error $\sim 1/\sqrt{N}$:** estimates tighten with more samples (verified).